# Ch14 Capstone v2 — GPT-2 small 推理（KV cache + FP16 WMMA + FlashAttention）

本 notebook 对应仓库 `code/ch14_mini_llm/mini_llm_adv2.cu`。
运行前请确认：Runtime → Change runtime type → GPU（T4 / A100 / L4 均可）。

In [6]:
# 1) 克隆仓库并自动选择 GPU 架构
import os, subprocess
if not os.path.exists('/content/ops'):
    !git clone https://github.com/hswsp/learn-cuda-from-scratch.git /content/ops
%cd /content/ops

gpu = subprocess.check_output('nvidia-smi --query-gpu=name --format=csv,noheader', shell=True).decode().strip()
print('detected GPU:', gpu)
ARCH = 'sm_75' if 'T4' in gpu else ('sm_89' if 'L4' in gpu else 'sm_80')
print('using ARCH =', ARCH)

/content/ops
detected GPU: Tesla T4
using ARCH = sm_75


## Step 1 — 下载权重
脚本会把 HuggingFace 的 `gpt2` 权重按固定二进制 layout 序列化，方便 C++ 一次 `fread`。
文件头是 16 个 `int32`（含 `n_layer / n_head / d_model` 等），后接 fp16 张量流。

In [7]:
!pip install -q transformers torch
!python data/download_gpt2.py --out data/gpt2-small.bin
# 输出: data/gpt2-small.bin (~250 MB, fp16)

[gpt2-dl] loading gpt2 ...
Loading weights: 100% 148/148 [00:00<00:00, 10952.54it/s]
[gpt2-dl] n_layer=12 n_head=12 d=768 vocab=50257 max_seq=1024
[gpt2-dl] wrote data/gpt2-small.bin  (248.9 MB)


## Step 2 — 编译与运行 v2
`mini_llm_adv2` 默认使用 `temperature=0.8`、`top_k=40`、`seed=42`，生成 64 个新 token。

In [8]:
!make -C code/ch14_mini_llm ARCH={ARCH} mini_llm_adv2

make: Entering directory '/content/ops/code/ch14_mini_llm'
make: 'mini_llm_adv2' is up to date.
make: Leaving directory '/content/ops/code/ch14_mini_llm'


## Step 3 — token ↔ 文本
本仓库为简化没在 C++ 里写 BPE tokenizer。用 Python 编码 / 解码：

In [9]:
from transformers import GPT2Tokenizer
tok = GPT2Tokenizer.from_pretrained('gpt2')

# 把输入文本转成 token id 列表
prompt = 'Hello, there is'
ids = tok.encode(prompt)
print(f'prompt: {repr(prompt)}')
print(f'ids: {ids}')
print(f'--tokens={",".join(map(str, ids))}')

prompt: 'Hello, there is'
ids: [15496, 11, 612, 318]
--tokens=15496,11,612,318


## 最终 Cell — 输入 `Hello, there is` 看 v2 推理返回什么

In [10]:
# 运行 v2 推理
import re, subprocess, shlex

tokens_arg = ','.join(map(str, ids))
cmd = (
    f'./code/ch14_mini_llm/mini_llm_adv2'
    f' --weights=data/gpt2-small.bin'
    f' --tokens={tokens_arg}'
    f' --max_new=64'
    f' --temperature=0.8'
    f' --top_k=40'
    f' --seed=42'
)
print('running:', cmd)
output = subprocess.check_output(cmd, shell=True, text=True, stderr=subprocess.STDOUT)
print(output)

# 从输出中抓取 final tokens
m = re.search(r'final tokens \((\d+)\):\s+([\d\s]+)', output)
if m:
    final_ids = [int(x) for x in m.group(2).strip().split()]
    text = tok.decode(final_ids)
    print('\n=== generated text ===')
    print(text)
else:
    print('could not parse final tokens')

running: ./code/ch14_mini_llm/mini_llm_adv2 --weights=data/gpt2-small.bin --tokens=15496,11,612,318 --max_new=64 --temperature=0.8 --top_k=40 --seed=42
[mini_llm_adv2] cfg: L=12 H=12 D=768 FF=3072 T_max=1024 V=50257
[mini_llm_adv2] weights loaded.
[mini_llm_adv2] prompt tokens (4): 15496 11 612 318 
[mini_llm_adv2] temperature=0.800 top_k=40 seed=42 max_new=64
[mini_llm_adv2] prefill -> 257
[mini_llm_adv2] step  1 -> 329
[mini_llm_adv2] step  2 -> 422
[mini_llm_adv2] step  3 -> 13
[mini_llm_adv2] step  4 -> 290
[mini_llm_adv2] step  5 -> 326
[mini_llm_adv2] step  6 -> 286
[mini_llm_adv2] step  7 -> 286
[mini_llm_adv2] step  8 -> 12
[mini_llm_adv2] step  9 -> 262
[mini_llm_adv2] step 10 -> 12
[mini_llm_adv2] step 11 -> 11
[mini_llm_adv2] step 12 -> 11
[mini_llm_adv2] step 13 -> 262
[mini_llm_adv2] step 14 -> 357
[mini_llm_adv2] step 15 -> 198
[mini_llm_adv2] step 16 -> 286
[mini_llm_adv2] step 17 -> 12
[mini_llm_adv2] step 18 -> 290
[mini_llm_adv2] step 19 -> 290
[mini_llm_adv2] step 20

In [ ]:
# 解码后的文本 hello, Ohio that not... 里，hello, 是输入，后面全部才是模型生成的